In [2]:
#Abrimos una sesión de spark
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("MySpark")
    .master("local[8]")                # Usar exactamente 8 cores
    .config("spark.sql.shuffle.partitions", "6")   # Particiones post-shuffle
    .config("spark.driver.memory", "4g")           # RAM del driver
    .config("spark.sql.adaptive.enabled", "true")  # Adaptive Query Execution
    .getOrCreate()
        )
# Verificar versión
print(spark.version)            # 3.x.x
print(spark.sparkContext.uiWebUrl)  # http://localhost:4040

3.5.0
http://d17fa7338063:4040


In [3]:
sc = spark.sparkContext

print(sc.appName)           # "MySpark"
print(sc.master)            # "local[4]"
print(sc.defaultParallelism) # Número de cores
print(sc.version)           # Versión de Spark

# ¡NUNCA hagas esto en producción!
# spark.stop()  # Mata toda la sesión

MySpark
local[8]
8
3.5.0


In [4]:
import pandas as pd

df_prod = pd.read_excel('data/senasica_prod_aguacate_hass_2001_2022.xlsx')

df_prod.head()

,Estado,Superficie_Ha,Volumen_Ton,Valor_MXN,Anio
0,Michoacán,78627.38,820223.82,4.505384e+09,2001
1,Baja California,21.00,49.40,2.964000e+05,2001
2,Sinaloa,9.00,0.00,0.000000e+00,2001
3,Michoacán,81895.25,792658.90,3.608521e+09,2002
4,Estado de México,1426.00,11280.00,5.517595e+07,2002


In [5]:
df_prod.to_csv('data/senasica_prod_aguacate.csv', index=False, encoding='utf-8')

In [20]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType,
    DoubleType, FloatType,
    BooleanType, DateType, TimestampType,
    ArrayType, MapType
)

# ── Esquema básico ─────────────────────────────────────
schema_prod = StructType([
    StructField("Estado",   StringType(),    nullable=True),
    StructField("Superficie_Ha",   DoubleType(),   nullable=True),
    StructField("Volumen_Ton",   DoubleType(),   nullable=True),
    StructField("Valor_MXN",   DoubleType(),   nullable=True),
    StructField("Anio",   IntegerType(),   nullable=True)
])

df_prod = spark.read \
    .option("header", "true") \
    .option("sep",",")  \
    .option("quote", '"') \
    .option("escape", '"') \
    .schema(schema_prod) \
    .option("nullValue", "N/A") \
    .option("encoding",     "UTF-8") \
    .option("nanValue",     "nan") \
    .csv("data/senasica_prod_aguacate.csv")

df_prod.printSchema()

root
 |-- Estado: string (nullable = true)
 |-- Superficie_Ha: double (nullable = true)
 |-- Volumen_Ton: double (nullable = true)
 |-- Valor_MXN: double (nullable = true)
 |-- Anio: integer (nullable = true)



In [21]:
df_prod.show()

+----------------+-------------+-----------------+---------------+----+
|          Estado|Superficie_Ha|      Volumen_Ton|      Valor_MXN|Anio|
+----------------+-------------+-----------------+---------------+----+
|       Michoacán|     78627.38|820223.8200000001|4.50538404147E9|2001|
| Baja California|         21.0|             49.4|       296400.0|2001|
|         Sinaloa|          9.0|              0.0|            0.0|2001|
|       Michoacán|     81895.25|         792658.9|3.60852063664E9|2002|
|Estado de México|       1426.0|          11280.0|   5.51759538E7|2002|
|         Morelos|        989.1|          9970.76|  4.635965239E7|2002|
| Baja California|         36.0|             42.5|       348500.0|2002|
|          Oaxaca|         36.0|            242.0|      1694000.0|2002|
|         Sinaloa|          9.0|             19.5|        58500.0|2002|
|       Michoacán|      82523.0|        800452.08|4.86958275885E9|2003|
|         Morelos|       2046.7|          19920.1|     8.946498E

In [22]:
# ── Contar nulls por columna ───────────────────────────
from pyspark.sql import functions as F

# El patrón profesional para auditar nulls
nulos_por_col = df_prod.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_prod.columns
])
nulos_por_col.show()

# ── Porcentaje de nulls ────────────────────────────────
total = df_prod.count()
pct_nulos = df_prod.select([
    F.round(
        F.count(F.when(F.col(c).isNull(), c)) / total * 100, 2
    ).alias(c)
    for c in df_prod.columns
])
pct_nulos.show()

+------+-------------+-----------+---------+----+
|Estado|Superficie_Ha|Volumen_Ton|Valor_MXN|Anio|
+------+-------------+-----------+---------+----+
|     0|            0|          0|        0|   0|
+------+-------------+-----------+---------+----+

+------+-------------+-----------+---------+----+
|Estado|Superficie_Ha|Volumen_Ton|Valor_MXN|Anio|
+------+-------------+-----------+---------+----+
|   0.0|          0.0|        0.0|      0.0| 0.0|
+------+-------------+-----------+---------+----+



In [23]:
df_prod.columns

['Estado', 'Superficie_Ha', 'Volumen_Ton', 'Valor_MXN', 'Anio']

In [24]:
#Como no hay datos nulos, procedemos a hacer un reporte estadístico inicial

from pyspark.sql.functions import max, min

#Cantidad de datos
print(f'Se analizan {df_prod.count()} filas con {len(df_prod.columns)} columnas')

#Qué años se analizan?
anios = df_prod.select('Anio').distinct().orderBy('Anio')
print(f'Están presentes {anios.count()} años')
print('Desde:  ')
# Calculamos y mostramos ambos valores al mismo tiempo
df_prod.agg(
    min("Anio").alias("inicio"),
    max("Anio").alias("final")
).show()


#Estadisticas y distribución de los datos
df_prod.select('Superficie_Ha', 'Volumen_Ton', 'Valor_MXN').summary().show()

Se analizan 404 filas con 5 columnas
Están presentes 23 años
Desde:  
+------+-----+
|inicio|final|
+------+-----+
|  2001| 2023|
+------+-----+

+-------+------------------+-----------------+--------------------+
|summary|     Superficie_Ha|      Volumen_Ton|           Valor_MXN|
+-------+------------------+-----------------+--------------------+
|  count|               404|              404|                 404|
|   mean|  8839.74198019802| 85030.3973143564|1.3388968508084402E9|
| stddev|29222.467147203773|300387.9480650845| 5.432515223571642E9|
|    min|               3.0|              0.0|                 0.0|
|    25%|              46.5|            213.0|          2471446.97|
|    50%|            528.76|           1947.0|       2.392235501E7|
|    75%|           2606.15|         15883.19|       1.776901112E8|
|    max|         178637.13|       2167472.78|   4.471851760907E10|
+-------+------------------+-----------------+--------------------+



In [25]:
from pyspark.sql.functions import col

df_prod = df_prod.withColumnRenamed("Superficie_Ha", "Superficie_miles_Ha")
df_prod = df_prod.withColumnRenamed("Volumen_Ton", "Volumen_miles_Ton")
df_prod = df_prod.withColumnRenamed("Valor_MXN", "Valor_millones_MXN")


# Multiplicar sobrescribiendo o creando una nueva columna
df_prod = df_prod.withColumn("Superficie_miles_Ha", col("Superficie_miles_Ha") / 1000)
df_prod = df_prod.withColumn("Volumen_miles_Ton", col("Volumen_miles_Ton") / 1000)
df_prod = df_prod.withColumn("Valor_millones_MXN", col("Valor_millones_MXN") / 1000000)


In [26]:
df_prod.select('Superficie_miles_Ha', 'Volumen_miles_Ton', 'Valor_millones_MXN').summary().show()

+-------+-------------------+------------------+------------------+
|summary|Superficie_miles_Ha| Volumen_miles_Ton|Valor_millones_MXN|
+-------+-------------------+------------------+------------------+
|  count|                404|               404|               404|
|   mean|  8.839741980198022| 85.03039731435638|1338.8968508084392|
| stddev| 29.222467147203787|300.38794806508463| 5432.515223571643|
|    min|              0.003|               0.0|               0.0|
|    25%|             0.0465|             0.213|        2.47144697|
|    50%|            0.52876|             1.947|       23.92235501|
|    75%|            2.60615|          15.88319|       177.6901112|
|    max|          178.63713|2167.4727799999996|    44718.51760907|
+-------+-------------------+------------------+------------------+



In [27]:
# Usamos coalesce(1) para forzar a Spark a juntar todo en un solo archivo .csv
df_prod.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv('output/senasica_prod_aguacate_limpio')

print("¡Reporte de serie de tiempo guardado exitosamente en CSV!")

¡Reporte de serie de tiempo guardado exitosamente en CSV!


In [ ]:
spark.stop()